<img src="https://raw.githubusercontent.com/tancao1128/ams261-labs/main/assets/lab21-banner.png" alt="AMS 261 Lab 21 · Fence Line: the highest level curve 4xy = k touches the fence x²/9 + y²/16 = 1, where grad f = λ grad g" width="100%">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tancao1128/ams261-labs/blob/main/labs/AMS261-Lecture21.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/tancao1128/ams261-labs/main?labpath=labs%2FAMS261-Lecture21.ipynb)
· [Download this notebook (.ipynb)](https://raw.githubusercontent.com/tancao1128/ams261-labs/main/labs/AMS261-Lecture21.ipynb)

## 0 · Mission briefing

**What if the summit is off-limits?** In Lecture 20 we found peaks by setting $\nabla f=\mathbf 0$.
Now the point $(x,y)$ must stay on a fence, a curve $g(x,y)=c$.
The unconstrained peak, if there is one, is not on the fence.
So we ask: where on the fence is $f$ largest?

Our fence is the ellipse $\dfrac{x^2}{9}+\dfrac{y^2}{16}=1$.
A rectangle is inscribed in it, with its corner $(x,y)$ in the first quadrant.
Its area is $f(x,y)=4xy$. Note that $4xy$ has no peak at all on the whole plane. It grows without bound.
The fence is what makes the problem have an answer. In this lab we find that answer in three ways.

- **Time:** about 30 minutes.
- **When:** after class, before Lecture 22.
- **Before you start:** read lecture §2–§4 and Example 1 on the Lecture 21 page. Skim Example 2.
- **Section numbers.** Sections are numbered 0–13. "(lecture §n)" points to the matching section of the Lecture 21 page.

**Lab objectives.** By the end of this lab you can

1. find a constrained maximum by sweeping the level curves $4xy=k$ until one is tangent to the fence;
2. check numerically and with sympy that $\nabla f=\lambda\nabla g$ holds exactly at that point;
3. confirm a two-constraint answer (Example 2) by scanning a parametrized circle, and read its multipliers as shadow prices.

**How to work.** Run the cells from top to bottom with Shift+Enter. A cell marked ✏️ asks you to type something first.

> **Not graded. Practice only.**

In [ ]:
# SETUP: run this cell first. It needs no local files.
import base64, json, math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
from IPython.display import HTML, Markdown, display

try:
    from ipywidgets import (FloatSlider, Button, Label, HBox, VBox, Layout,
                            HTML as WidgetHTML, interactive_output)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

x, y, z = sp.symbols('x y z', real=True)
lam, mu = sp.symbols('lambda mu', real=True)

# Colour tokens. Each colour has one meaning in every plot of this lab.
INK = '#1E293B'        # text, the tangent level curve, points on the fence
GRAD = '#2563EB'       # grad f, and maximum points
LOW = '#DC2626'        # minimum points
TANGENT = '#059669'    # the tangent line of the fence
HIGHLIGHT = '#D97706'  # grad g, and the level curve you are moving
CONTOUR = '#94A3B8'    # other level curves of f
FENCE = '#7C3AED'      # the constraint curve (the fence)

# Console tokens. They are used only for the frame around each plot.
HUD_NAVY = '#0A0E14'   # top and bottom bars
HUD_PANEL = '#0F1A2B'  # the outer frame
CYAN = '#22D3EE'       # frame text and corner brackets
SCREEN = '#F8FAFC'     # the light screen behind every panel

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})


def show_math(latex):
    # Display one formula.
    display(Markdown(f"$${latex}$$"))


def show_text(text):
    # Display formatted text.
    display(Markdown(text))


def vec(v):
    # LaTeX for a vector written with angle brackets.
    return r'\left\langle ' + ',\ '.join(sp.latex(c) for c in v) + r'\right\rangle'


def fmt(v, digits=2):
    # Format a number; never print -0.00.
    v = float(v)
    if abs(v) < 0.5 * 10 ** (-digits):
        v = 0.0
    return f"{v:.{digits}f}"


def draw_arrow(ax, start, vec, color, ls='-', lw=2.2):
    # Draw one arrow from start along vec.
    ax.annotate('', xy=(start[0] + vec[0], start[1] + vec[1]), xytext=(start[0], start[1]),
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw, ls=ls, mutation_scale=16))


def legend_line(color, label, ls='-', marker=None):
    # A legend entry for an arrow or a marker.
    return Line2D([0], [0], color=color, ls=ls, lw=2.2 if marker is None else 0,
                  marker=marker, markersize=8, label=label)


def as_number(v):
    # Accept a number or a string such as "3/sqrt(2)".
    return float(sp.sympify(v))


def check_answer(given, target, tol=5e-3):
    # Compare an answer with a target. Returns True or False; never raises.
    try:
        if given is None:
            return False
        if isinstance(target, bool):
            return given is target
        if isinstance(target, (list, tuple)):
            return (len(given) == len(target)
                    and all(check_answer(a, b, tol) for a, b in zip(given, target)))
        return math.isclose(as_number(given), float(target), rel_tol=1e-3, abs_tol=tol)
    except Exception:
        return False


def hud_frame(fig, label, footer='FENCE LINE · f = 4xy · fence x²/9 + y²/16 = 1'):
    # Put a finished figure inside a navy console frame.
    # All panels, labels and legends sit on one light screen, so axis text stays dark on light.
    side, top, foot, gap, pad = 0.30, 0.40, 0.30, 0.10, 0.12   # sizes in inches
    w, h = fig.get_size_inches()
    W, H = w + 2 * (side + pad), h + top + foot + 2 * (gap + pad)
    fig.set_size_inches(W, H)
    fig.patch.set_facecolor(HUD_PANEL)

    # Dark ink on the light screen.
    for ax in fig.axes:
        ax.set_facecolor(SCREEN)
        ax.tick_params(colors=INK, labelcolor=INK)
        ax.title.set_color(INK)
        ax.xaxis.label.set_color(INK)
        ax.yaxis.label.set_color(INK)
        for spine in ax.spines.values():
            spine.set_color('#64748B')
        if ax.get_legend() is not None:
            for t in ax.get_legend().get_texts():
                t.set_color(INK)

    # The screen, and the area inside it where the panels go.
    sx0, sy0, sx1, sy1 = side, foot + gap, W - side, H - top - gap   # in inches
    box = [sx0 + pad, sy0 + pad, sx1 - pad, sy1 - pad]
    for _ in range(3):
        fig.tight_layout(rect=(box[0] / W, box[1] / H, box[2] / W, box[3] / H))
        bb = fig.get_tightbbox(fig.canvas.get_renderer())   # everything drawn so far, in inches
        over = [sx0 + pad - bb.x0, sy0 + pad - bb.y0, bb.x1 - sx1 + pad, bb.y1 - sy1 + pad]
        if max(over) < 0.01:
            break   # labels and legends all fit on the screen
        box = [box[0] + max(over[0], 0), box[1] + max(over[1], 0),
               box[2] - max(over[2], 0), box[3] - max(over[3], 0)]
    fig.add_artist(Rectangle((sx0 / W, sy0 / H), (sx1 - sx0) / W, (sy1 - sy0) / H,
                             transform=fig.transFigure, facecolor=SCREEN,
                             edgecolor='#334155', lw=1, zorder=-10))

    # A faint grid on the screen, about 24 px apart.
    step = 0.24
    segs = [[(a / W, sy0 / H), (a / W, sy1 / H)] for a in np.arange(sx0 + step, sx1, step)]
    segs += [[(sx0 / W, b / H), (sx1 / W, b / H)] for b in np.arange(sy0 + step, sy1, step)]
    fig.add_artist(LineCollection(segs, colors='#E2E8F0', linewidths=0.6,
                                  transform=fig.transFigure, zorder=-9))

    # Top bar with the section label, and a small footer bar.
    for y_bar, height in [(H - top, top), (0, foot)]:
        fig.add_artist(Rectangle((0, y_bar / H), 1, height / H, transform=fig.transFigure,
                                 facecolor=HUD_NAVY, edgecolor='none', zorder=-8))
    fig.add_artist(Line2D([0, 1], [(H - top) / H] * 2, transform=fig.transFigure, color=CYAN, lw=1.5))
    fig.add_artist(Line2D([0, 1], [foot / H] * 2, transform=fig.transFigure, color=CYAN, lw=0.8, alpha=0.6))
    mono = dict(family='monospace', transform=fig.transFigure, va='center')
    fig.text(side / W, 1 - top / (2 * H), 'AMS 261 · M21 · ' + label, color=CYAN,
             fontsize=10, fontweight='bold', ha='left', **mono)
    fig.text(1 - side / W, 1 - top / (2 * H), '● FENCE LINK · LIVE', color=CYAN,
             fontsize=8, ha='right', **mono)
    fig.text(side / W, foot / (2 * H), footer, color=CONTOUR, fontsize=8, ha='left', **mono)
    fig.text(1 - side / W, foot / (2 * H), 'MISSION 21', color=CONTOUR, fontsize=8, ha='right', **mono)

    # Cyan corner brackets just outside the screen.
    off, arm = 0.07, 0.24
    for cx, cy, dx, dy in [(sx0 - off, sy0 - off, 1, 1), (sx1 + off, sy0 - off, -1, 1),
                           (sx0 - off, sy1 + off, 1, -1), (sx1 + off, sy1 + off, -1, -1)]:
        fig.add_artist(Line2D([(cx + dx * arm) / W, cx / W, cx / W],
                              [cy / H, cy / H, (cy + dy * arm) / H],
                              transform=fig.transFigure, color=CYAN, lw=2))
    return fig


CARD = ('max-width:560px;margin-top:6px;border:1px solid #0F1A2B;border-radius:10px;overflow:hidden;'
        'background:#F8FAFC;color:#1E293B;font-family:system-ui,-apple-system,Segoe UI,sans-serif;')
CARD_HEAD = ('background:#0F1A2B;color:#22D3EE;padding:6px 14px;border-bottom:2px solid #22D3EE;'
             'font-family:Menlo,Consolas,monospace;letter-spacing:2px;font-size:12px;')


def check_card(title, rows, summary):
    # A small result card. Each row is (item, mark, words), for example ('k_max', '✅', 'correct').
    body = ''.join(
        f'<div style="padding:3px 0;color:#1E293B;">'
        f'<span style="display:inline-block;min-width:330px;padding-right:12px;font-family:Menlo,Consolas,monospace;">{item}</span>'
        f'{mark} {words}</div>' for item, mark, words in rows)
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">{title}</div>'
                 f'<div style="padding:8px 14px;color:#1E293B;">{body}'
                 f'<div style="margin-top:6px;font-weight:600;color:#1E293B;">{summary}</div></div></div>'))


def mission_complete_card():
    # Shown once the mission log and the retrieval check are both correct.
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">✅ MISSION 21 COMPLETE</div>'
                 '<div style="padding:8px 14px;line-height:1.5;color:#1E293B;">'
                 'Your mission log and your retrieval check are both correct.<br>'
                 '<b>Prediction for Lecture 22.</b> Chapter 14 asks how to add up f over a region '
                 'instead of differentiating it. How would you estimate the area of the ellipse '
                 'from lecture §4 by slicing it into thin strips?<br>'
                 'Write your idea in one sentence. Lecture 22 will test it.</div></div>'))


def lagrange_system(objective, constraints, variables, multipliers):
    # The course-notebook solver: L = f - sum(multiplier * constraint).
    # Each constraint is written as an expression that must equal 0.
    L = objective - sum(m * c for m, c in zip(multipliers, constraints))
    equations = [sp.diff(L, var) for var in variables] + list(constraints)
    solutions = sp.solve(equations, list(variables) + list(multipliers), dict=True)
    values = [sp.simplify(objective.subs(s)) for s in solutions]
    return L, equations, solutions, values


print('Setup complete. Widgets available:', HAS_WIDGETS)

---
## 1 · PREDICT · Where does the hyperbola touch? (lecture §2)

🔮 **STAGE 1/7 · PREDICT** ▰▱▱▱▱▱▱

The problem reads:

$$
\begin{cases}\text{maximize} & f(x,y)=4xy,\\ \text{subject to} & g(x,y)=\dfrac{x^2}{9}+\dfrac{y^2}{16}=1,\quad x>0,\ y>0.\end{cases}
$$

The level curves $4xy=k$ are the hyperbolas $y=k/(4x)$. As $k$ grows, the hyperbola moves outward.
**Do not compute yet.** Make two guesses.

1. What is the largest $k$ for which the hyperbola $4xy=k$ still touches the fence?
2. At that contact point, what is the $x$-coordinate of the corner?

A rough guess is fine. The fence fits inside the box $0\le x\le 3$, $0\le y\le 4$. You will check your guess in Section 4.

In [ ]:
# ✏️ PREDICT. Type your two guesses, then run this cell. Do not compute first.
my_k_guess = None    # the largest k for which 4xy = k still touches the fence
my_x_guess = None    # the x-coordinate of the contact point (between 0 and 3)

if my_k_guess is None or my_x_guess is None:
    print('Prediction not logged yet. Type two guesses above and run this cell again.')
else:
    print(f'Prediction logged: k = {my_k_guess}, x = {my_x_guess}.')
    print('You will check it in Section 4.')

In [ ]:
# The model of Example 1. The fence is g(x, y) = c with c = 1.
f_expr = 4*x*y
g_expr = x**2/9 + y**2/16
c_fence = 1

# The fence can be traced as (3 cos t, 4 sin t). Check that this point is on the fence.
t = sp.symbols('t', real=True)
on_fence = sp.simplify(g_expr.subs({x: 3*sp.cos(t), y: 4*sp.sin(t)}) - c_fence)
print('(3 cos t, 4 sin t) lies on the fence for every t:', on_fence == 0)

# Numerical copies for the plots.
f_num = sp.lambdify((x, y), f_expr, 'numpy')
grad_f_num = sp.lambdify((x, y), [sp.diff(f_expr, x), sp.diff(f_expr, y)], 'numpy')
grad_g_num = sp.lambdify((x, y), [sp.diff(g_expr, x), sp.diff(g_expr, y)], 'numpy')

---
## 2 · EXPLORE · Level-curve sweep (lecture §2)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

**Idea from the page.** Sweep the level curves $4xy=k$ outward. The largest $k$ for which the hyperbola still touches the fence is reached when the two curves are **tangent**.

**Your task.** Move the slider $k$ from small to large. Watch the readout under the plot. It counts the contact points of the hyperbola with the first-quadrant arc of the fence.

- With **2** contact points, the hyperbola cuts the fence. A larger $k$ is still possible.
- With **0** contact points, the hyperbola has left the fence. This $k$ is too large.
- Find the value of $k$ where the count drops to **1**. Write it down.

The dashed rectangle is the inscribed rectangle built on the first contact point. Its area is $k$.

If the slider does not appear, use the static cell after it. Change `k_static` there and run it again.

In [ ]:
# Values of f along the first-quadrant arc, used to count contact points.
t_arc = np.linspace(1e-6, np.pi/2 - 1e-6, 20001)
x_arc, y_arc = 3*np.cos(t_arc), 4*np.sin(t_arc)
f_arc = f_num(x_arc, y_arc)
TOUCH_TOL = 0.05     # |k - (largest value on the arc)| below this counts as touching


def contact_points(k):
    # The points of the arc where 4xy = k, found from sign changes of f - k.
    if abs(k - f_arc.max()) <= TOUCH_TOL:
        i = int(np.argmax(f_arc))
        return [(x_arc[i], y_arc[i])], 'tangent'
    d = f_arc - k
    idx = np.where(np.sign(d[:-1]) != np.sign(d[1:]))[0]
    pts = []
    for i in idx:   # linear interpolation between the two samples
        s = d[i] / (d[i] - d[i + 1])
        pts.append((x_arc[i] + s*(x_arc[i+1] - x_arc[i]), y_arc[i] + s*(y_arc[i+1] - y_arc[i])))
    return pts, ('crossing' if pts else 'no contact')


def plot_sweep(k=8.0):
    pts, kind = contact_points(k)
    fig, ax = plt.subplots(figsize=(10.5, 7))
    s = np.linspace(0, 2*np.pi, 400)
    ax.plot(3*np.cos(s), 4*np.sin(s), color=FENCE, lw=1.2, alpha=0.5)
    ax.plot(x_arc, y_arc, color=FENCE, lw=3)
    xs = np.linspace(0.05, 5.5, 600)
    for kr in (8, 16, 32):                      # three fixed reference level curves
        ax.plot(xs, kr/(4*xs), color=CONTOUR, lw=1)
    if k > 0:
        ax.plot(xs, k/(4*xs), color=HIGHLIGHT, lw=2.6)
    for (px, py) in pts:
        ax.plot([px], [py], 'o', color=INK, ms=8)
    if pts:
        px, py = pts[0]
        ax.plot([-px, px, px, -px, -px], [-py, -py, py, py, -py], color=INK, ls='--', lw=1.2)
    ax.set_xlim(-3.6, 5.5)
    ax.set_ylim(-4.6, 5.5)
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'k = {k:.2f} · contact points: {len(pts)} ({kind})')
    ax.legend(handles=[
        legend_line(FENCE, 'purple: the fence x²/9 + y²/16 = 1 (thick: x, y > 0)'),
        legend_line(HIGHLIGHT, 'amber: the level curve 4xy = k you are moving'),
        legend_line(CONTOUR, 'grey: fixed level curves 4xy = 8, 16, 32'),
        legend_line(INK, 'black dots: contact points with the arc', marker='o'),
        legend_line(INK, 'black dashed: rectangle on the first contact point', ls='--'),
    ], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
    hud_frame(fig, 'SEC 2 · LEVEL-CURVE SWEEP')
    display(fig)
    plt.close(fig)


if HAS_WIDGETS:
    k_slider = FloatSlider(min=0, max=40, step=0.05, value=8, description='k',
                           readout_format='.2f', continuous_update=False,
                           layout=Layout(width='100%'),
                           style={'handle_color': CYAN, 'description_width': 'initial'})
    sweep_readout = WidgetHTML()

    def update_sweep_readout(change=None):
        pts, kind = contact_points(k_slider.value)
        sweep_readout.value = (f'<span style="font-family:Menlo,Consolas,monospace;">READOUT · '
                               f'k = {k_slider.value:.2f} · contact points: {len(pts)} · {kind}</span>')

    k_slider.observe(update_sweep_readout, names='value')
    update_sweep_readout()
    display(VBox([k_slider, interactive_output(plot_sweep, {'k': k_slider}), sweep_readout]))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* The purple ellipse is the fence. Its first-quadrant arc is drawn thick. The amber hyperbola is the level curve $4xy=k$ for the slider value. Three grey hyperbolas are fixed level curves for comparison. Black dots mark where the amber curve meets the arc. A black dashed rectangle is inscribed in the fence with its corner at the first contact point. The title prints $k$ and the number of contact points.

In [ ]:
# Static version (no widgets needed). Change k_static and run again.
k_static = 16
plot_sweep(k_static)

*Figure description.* The same plot, drawn once at the value of $k$ stored in `k_static`.

---
## 3 · EXPLORE · Gradient alignment (lecture §3)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

The page turns "tangent" into an equation. Since $\nabla f$ is normal to the level curve of $f$ and $\nabla g$ is normal to the fence, tangency means

$$
\nabla f(x_0,y_0)=\lambda\,\nabla g(x_0,y_0)\quad\text{for some scalar }\lambda .
$$

Here $\nabla f=\langle 4y,\,4x\rangle$ and $\nabla g=\left\langle \tfrac{2x}{9},\,\tfrac{y}{8}\right\rangle$.

**Your task.** Walk along the fence with the slider. The point is $(3\cos t,\ 4\sin t)$, with $t$ in degrees.
The plot draws the directions of $\nabla f$ and $\nabla g$ at the point. The readout prints the signed angle from $\nabla g$ to $\nabla f$.

- Find the value of $t$ where the angle is $0^\circ$. Read the point $(x,y)$ there.
- At that point, what does the level curve of $f$ do relative to the fence?

If the slider does not appear, use the static cell after it.

In [ ]:
def alignment(t_deg):
    # The point on the fence, both gradients, and the signed angle from grad g to grad f.
    tr = np.radians(t_deg)
    px, py = 3*np.cos(tr), 4*np.sin(tr)
    gf = np.array(grad_f_num(px, py), dtype=float)
    gg = np.array(grad_g_num(px, py), dtype=float)
    angle = np.degrees(np.arctan2(gg[0]*gf[1] - gg[1]*gf[0], gg @ gf))
    return px, py, gf, gg, angle


def plot_alignment(t_deg=20.0):
    px, py, gf, gg, angle = alignment(t_deg)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5.0), gridspec_kw={'width_ratios': [1, 1.1]})

    # Left: the arc, the level curve of f through the point, and the two gradient directions.
    ax1.plot(x_arc, y_arc, color=FENCE, lw=3)
    xs = np.linspace(0.05, 5.0, 500)
    ax1.plot(xs, f_num(px, py)/(4*xs), color=CONTOUR, lw=1.6)
    tan = np.array([-gg[1], gg[0]]) / np.hypot(*gg)          # tangent of the fence
    ax1.plot([px - 1.4*tan[0], px + 1.4*tan[0]], [py - 1.4*tan[1], py + 1.4*tan[1]],
             color=TANGENT, ls='--', lw=1.8)
    draw_arrow(ax1, (px, py), 1.5*gf/np.hypot(*gf), GRAD, lw=2.6)
    draw_arrow(ax1, (px, py), 1.0*gg/np.hypot(*gg), HIGHLIGHT, lw=2.6)
    ax1.plot([px], [py], 'o', color=INK, ms=8)
    ax1.set_xlim(0, 5.0)
    ax1.set_ylim(0, 5.5)
    ax1.set_aspect('equal')
    ax1.grid(alpha=0.25)
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_title(f't = {t_deg:.1f}°, point ({px:.4f}, {py:.4f})')
    ax1.legend(handles=[
        legend_line(FENCE, 'purple: the fence (x, y > 0)'),
        legend_line(CONTOUR, 'grey: the level curve of f through the point'),
        legend_line(TANGENT, 'green dashed: tangent line of the fence', ls='--'),
        legend_line(GRAD, r'blue: direction of $\nabla f$'),
        legend_line(HIGHLIGHT, r'amber: direction of $\nabla g$'),
    ], loc='lower left', fontsize=8.5, framealpha=0.9)

    # Right: the signed angle from grad g to grad f along the whole arc.
    ts = np.linspace(1, 89, 400)
    ax2.plot(ts, [alignment(tt)[4] for tt in ts], color=INK, lw=2,
             label=r'signed angle from $\nabla g$ to $\nabla f$')
    ax2.axhline(0, color=CONTOUR, lw=1)
    ax2.plot([t_deg], [angle], 'o', color=HIGHLIGHT, ms=10, label='current point (amber dot)')
    ax2.set_xlim(0, 90)
    ax2.set_xticks(range(0, 91, 15))
    ax2.set_xlabel('t (degrees) on the fence (3 cos t, 4 sin t)')
    ax2.set_ylabel('angle (degrees)')
    ax2.set_title(f'angle = {angle:.2f}°')
    ax2.grid(alpha=0.3)
    ax2.legend(loc='upper right', fontsize=9)
    hud_frame(fig, 'SEC 3 · GRADIENT ALIGNMENT')
    display(fig)
    plt.close(fig)


if HAS_WIDGETS:
    t_slider = FloatSlider(min=1, max=89, step=0.1, value=20, description='t (deg)',
                           readout_format='.1f', continuous_update=False,
                           layout=Layout(width='100%'),
                           style={'handle_color': CYAN, 'description_width': 'initial'})
    align_readout = WidgetHTML()

    def update_align_readout(change=None):
        px, py, gf, gg, angle = alignment(t_slider.value)
        align_readout.value = (f'<span style="font-family:Menlo,Consolas,monospace;">READOUT · '
                               f't = {t_slider.value:.1f}° · point ({px:.4f}, {py:.4f}) · '
                               f'angle from ∇g to ∇f = {angle:.2f}°</span>')

    t_slider.observe(update_align_readout, names='value')
    update_align_readout()
    display(VBox([t_slider, interactive_output(plot_alignment, {'t_deg': t_slider}), align_readout]))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* Left: the purple arc of the fence and a black point on it. The grey hyperbola is the level curve of $f$ through the point. The green dashed line is the tangent line of the fence. The blue arrow shows the direction of $\nabla f$, and the amber arrow shows the direction of $\nabla g$. Both arrows are drawn with fixed lengths, so only their directions matter. Right: the signed angle from $\nabla g$ to $\nabla f$ along the whole arc. An amber dot marks the current point.

In [ ]:
# Static version (no widgets needed). Change t_static and run again.
t_static = 60
plot_alignment(t_static)

*Figure description.* The same two panels, drawn once at the value of $t$ stored in `t_static`.

---
## 4 · EXPLAIN · The Lagrange system (lecture §4)

🧠 **STAGE 3/7 · EXPLAIN** ▰▰▰▱▱▱▱

**Theorem (one constraint, lecture §3).** Let $f$ and $g$ have continuous first partial derivatives. If $f$ has an extremum at $(x_0,y_0)$ on the curve $g(x,y)=c$ and $\nabla g(x_0,y_0)\neq\mathbf 0$, then there is a scalar $\lambda$ such that $\nabla f(x_0,y_0)=\lambda\,\nabla g(x_0,y_0)$.

So we solve three equations in three unknowns $(x,y,\lambda)$:

$$
\begin{cases} f_x=\lambda\, g_x,\\ f_y=\lambda\, g_y,\\ g(x,y)=c.\end{cases}
$$

The cell below solves this system with the course solver. Then it checks three things:
the gradients are parallel, the regularity hypothesis $\nabla g\neq\mathbf 0$ holds, and a numerical scan of the arc agrees.

In [ ]:
L1, eqs1, sols1, vals1 = lagrange_system(f_expr, [g_expr - c_fence], [x, y], [lam])
show_math(r'\mathcal L(x,y,\lambda) = ' + sp.latex(L1))
show_text('**Equations** (each one equals 0):')
show_math(r',\qquad '.join(sp.latex(e) for e in eqs1))

# Keep the solution with x > 0 and y > 0.
sol1 = [s for s in sols1 if s[x] > 0 and s[y] > 0][0]
x_star, y_star, lam_star = sol1[x], sol1[y], sol1[lam]
f_star = sp.simplify(f_expr.subs(sol1))
show_math(r'(x^*,y^*) = \left(' + sp.latex(x_star) + ',\ ' + sp.latex(y_star) + r'\right)\approx('
          + fmt(x_star, 4) + ',\ ' + fmt(y_star, 4) + r'),\qquad \lambda^* = ' + sp.latex(lam_star)
          + r',\qquad f(x^*,y^*) = ' + sp.latex(f_star))

# Check 1: grad f - lambda grad g = 0 at the point.
gf_star = sp.Matrix([sp.diff(f_expr, v) for v in (x, y)]).subs(sol1)
gg_star = sp.Matrix([sp.diff(g_expr, v) for v in (x, y)]).subs(sol1)
show_math(r'\nabla f(x^*,y^*) = ' + vec(gf_star) + r',\qquad \nabla g(x^*,y^*) = ' + vec(gg_star))
print('grad f = lambda grad g holds exactly:', sp.simplify(gf_star - lam_star*gg_star) == sp.zeros(2, 1))

# Check 2: regularity. grad g = 0 only at the origin, and the origin is not on the fence.
crit_g = sp.solve([sp.diff(g_expr, x), sp.diff(g_expr, y)], [x, y], dict=True)
print('grad g = 0 only at', crit_g, '| on the fence?', [g_expr.subs(s) == c_fence for s in crit_g])

# Check 3: a numerical scan of f along the arc.
i_best = int(np.argmax(f_arc))
print(f'Scan of the arc: largest f = {f_arc[i_best]:.4f} at ({x_arc[i_best]:.4f}, {y_arc[i_best]:.4f}).')

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🧠 Explain.</strong> Compare three numbers: the $k$ where the contact count dropped to 1 in Section 2, the value $f(x^*,y^*)$, and the point where the angle was $0^\circ$ in Section 3.
<br><br>Now explain <b>in two sentences</b>: why must the level curve $4xy=k$ be tangent to the fence at the maximum? Why does tangency of the two curves mean $\nabla f=\lambda\nabla g$?
<br><br>The page calls a multiplier a <b>shadow price</b>. You will test what that means on Example 2 in Section 6.1.

</div>

Write your sentences first. Then run the next cell to compare them with your Section 1 prediction.

In [ ]:
# Check your Section 1 prediction.
print(f'Largest k that touches the fence: {fmt(f_star, 4)}.')
print(f'Contact point: x* = {fmt(x_star, 4)}, y* = {fmt(y_star, 4)}.')
if my_k_guess is None or my_x_guess is None:
    print('You did not log a prediction in Section 1. Next time, commit a guess first.')
else:
    try:
        print(f'Your k guess was off by {abs(float(my_k_guess) - float(f_star)):.2f}.')
        print(f'Your x guess was off by {abs(float(my_x_guess) - float(x_star)):.3f}.')
    except (TypeError, ValueError):
        print('Log your guesses as numbers, then run this cell again.')

---
## 5 · CHALLENGE · The whole fence (lecture §3)

🎯 **STAGE 4/7 · CHALLENGE** ▰▰▰▰▱▱▱

Now drop the condition $x>0,\ y>0$. We look at $f(x,y)=4xy$ on the **whole** ellipse $\dfrac{x^2}{9}+\dfrac{y^2}{16}=1$.
This is no longer an area. It is just a function on a closed curve.

Before you run the cell, answer two questions.

1. How many solutions should the Lagrange system have now?
2. The page solves Example 1 by writing $\lambda=18y/x$. This divides by $x$. Which points of the fence does that step skip?

The cell solves the system on the whole fence. Then it checks the points with $x=0$ by hand.

In [ ]:
show_text('**All solutions of the Lagrange system on the whole fence:**')
print(f"{'x':>10}{'y':>10}{'lambda':>10}{'f':>8}")
for s in sorted(sols1, key=lambda s: (float(s[x]), float(s[y]))):
    print(f"{fmt(s[x], 4):>10}{fmt(s[y], 4):>10}{str(s[lam]):>10}{str(sp.simplify(f_expr.subs(s))):>8}")

# The points skipped by dividing by x: x = 0 on the fence gives y = 4 or y = -4.
show_text('**The points with x = 0, checked by hand:**')
for py in (4, -4):
    gfv = sp.Matrix([sp.diff(f_expr, v) for v in (x, y)]).subs({x: 0, y: py})
    ggv = sp.Matrix([sp.diff(g_expr, v) for v in (x, y)]).subs({x: 0, y: py})
    parallel = sp.simplify(gfv[0]*ggv[1] - gfv[1]*ggv[0]) == 0
    print(f'(0, {py}): grad f = {list(gfv)}, grad g = {list(ggv)}, parallel: {parallel}, f = {f_expr.subs({x: 0, y: py})}')

In [ ]:
# f along the whole fence (3 cos t, 4 sin t), with the Lagrange points marked.
ts = np.linspace(0, 360, 1441)
f_loop = f_num(3*np.cos(np.radians(ts)), 4*np.sin(np.radians(ts)))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.plot(ts, f_loop, color=FENCE, lw=2.2)
ax.axhline(0, color=CONTOUR, lw=1)
for s in sols1:
    t_s = math.degrees(math.atan2(float(s[y])/4, float(s[x])/3)) % 360
    val = float(f_expr.subs(s))
    ax.plot([t_s], [val], 'o', color=GRAD if val > 0 else LOW, ms=10)
for t_s in (90, 270):
    ax.plot([t_s], [0], 's', color=INK, ms=8)
ax.set_xlim(0, 360)
ax.set_xticks(range(0, 361, 45))
ax.set_xlabel('t (degrees) on the fence (3 cos t, 4 sin t)')
ax.set_ylabel('f = 4xy')
ax.set_title('Section 5 · f along the whole fence, with every Lagrange point')
ax.grid(alpha=0.3)
ax.legend(handles=[
    legend_line(FENCE, 'purple: f along the whole fence'),
    legend_line(GRAD, 'blue dots: Lagrange points with lambda > 0', marker='o'),
    legend_line(LOW, 'red dots: Lagrange points with lambda < 0', marker='o'),
    legend_line(INK, 'black squares: the points x = 0, checked by hand', marker='s'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 5 · THE WHOLE FENCE')
plt.show()

*Figure description.* The purple wave is $f=4xy$ along the whole fence, plotted against the angle $t$. Blue dots sit on the two peaks and red dots on the two valleys. These are the four solutions of the Lagrange system. Black squares mark the two points with $x=0$, at $t=90^\circ$ and $t=270^\circ$. The wave crosses zero there and has no peak or valley.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🎯 Challenge question.</strong> The system has four solutions. Two give $f=24$ and two give $f=-24$. The system itself does not say which are maxima. What step decides this? Also, the points $(0,\pm4)$ were skipped when we divided by $x$. Why is it safe to drop them here? Answer in two or three sentences.

</div>

<details>
<summary>Hint (open after you try)</summary>

The Lagrange system gives candidates only. We compare the values of $f$ at all candidates. At $(0,\pm4)$, check whether $\nabla f$ and $\nabla g$ are parallel. If they are not, the theorem says the point cannot be an extremum, since $\nabla g\neq\mathbf 0$ there.

</details>

---
## 6 · TRANSFER · Sphere meets plane (lecture §6)

🚀 **STAGE 5/7 · TRANSFER** ▰▰▰▰▰▱▱

**Page Example 2.** The temperature is $T(x,y,z)=20+2x+2y+z^2$. We want its extreme values on the curve where the sphere $x^2+y^2+z^2=11$ meets the plane $x+y+z=3$.
The page solves it with two multipliers. With $g=x^2+y^2+z^2$ and $h=x+y+z$, the theorem (lecture §5) assumes that $\nabla g$ and $\nabla h$ are linearly independent at the candidate. Then $\nabla T=\lambda\nabla g+\mu\nabla h$ there.
It finds $T_{\min}=25$ at $(3,-1,1)$ and $(-1,3,1)$, and $T_{\max}=\tfrac{91}{3}$ at $\left(1\pm\tfrac{2}{\sqrt3},\,1\pm\tfrac{2}{\sqrt3},\,1\mp\tfrac{4}{\sqrt3}\right)$.

Here we check this answer **without** the Lagrange system. The curve is a circle, so we walk around it and record $T$.

- The point of the plane closest to the origin is $C=(1,1,1)$. Its distance from the origin is $\sqrt3$.
- So the circle has centre $C$ and radius $r=\sqrt{11-3}=2\sqrt2$.
- Two unit vectors in the plane, perpendicular to each other, are $\mathbf e_1=\tfrac{1}{\sqrt2}\langle1,-1,0\rangle$ and $\mathbf e_2=\tfrac{1}{\sqrt6}\langle1,1,-2\rangle$.

The circle is $\mathbf r(s)=C+2\sqrt2\,(\cos s\,\mathbf e_1+\sin s\,\mathbf e_2)$ for $0\le s<2\pi$.

**Predict first.** On which part of the circle do you expect it to be hottest: where $z$ is large, or where $x+y$ is large?

In [ ]:
s_sym = sp.symbols('s', real=True)
C0 = sp.Matrix([1, 1, 1])
e1 = sp.Matrix([1, -1, 0]) / sp.sqrt(2)
e2 = sp.Matrix([1, 1, -2]) / sp.sqrt(6)
r_s = C0 + 2*sp.sqrt(2)*(sp.cos(s_sym)*e1 + sp.sin(s_sym)*e2)

T_expr = 20 + 2*x + 2*y + z**2
g2, h2 = x**2 + y**2 + z**2, x + y + z
sub_s = {x: r_s[0], y: r_s[1], z: r_s[2]}
print('r(s) is on the sphere for every s:', sp.simplify(g2.subs(sub_s) - 11) == 0)
print('r(s) is on the plane  for every s:', sp.simplify(h2.subs(sub_s) - 3) == 0)


def circle_scan(c_g=11.0, c_h=3.0, n=200001):
    # Sample T on the circle where x^2 + y^2 + z^2 = c_g meets x + y + z = c_h.
    ss = np.linspace(0, 2*np.pi, n, endpoint=False)
    centre = (c_h/3) * np.ones(3)
    radius = np.sqrt(c_g - c_h**2/3)
    E1 = np.array([1.0, -1.0, 0.0]) / np.sqrt(2)
    E2 = np.array([1.0, 1.0, -2.0]) / np.sqrt(6)
    pts = centre[:, None] + radius*(np.outer(E1, np.cos(ss)) + np.outer(E2, np.sin(ss)))
    return ss, pts, 20 + 2*pts[0] + 2*pts[1] + pts[2]**2


ss, pts, T_vals = circle_scan()
i_min, i_max = int(np.argmin(T_vals)), int(np.argmax(T_vals))
print(f'Scan: T_min = {T_vals[i_min]:.6f} at ({pts[0, i_min]:.4f}, {pts[1, i_min]:.4f}, {pts[2, i_min]:.4f})')
print(f'Scan: T_max = {T_vals[i_max]:.6f} at ({pts[0, i_max]:.4f}, {pts[1, i_max]:.4f}, {pts[2, i_max]:.4f})')
print(f'Page: T_min = 25,  T_max = 91/3 = {91/3:.6f}')
print('Scan agrees with the page:', abs(T_vals[i_min] - 25) < 1e-6 and abs(T_vals[i_max] - 91/3) < 1e-6)

In [ ]:
fig = plt.figure(figsize=(12.5, 5.2))
ax3 = fig.add_subplot(1, 2, 1, projection='3d')
u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 20))
R = math.sqrt(11)
ax3.plot_wireframe(R*np.cos(u)*np.sin(v), R*np.sin(u)*np.sin(v), R*np.cos(v),
                   color=CONTOUR, lw=0.4, alpha=0.5)
ax3.plot(pts[0, ::200], pts[1, ::200], pts[2, ::200], color=FENCE, lw=2.6)
page_min = [(3, -1, 1), (-1, 3, 1)]
page_max = [(1 + 2/math.sqrt(3), 1 + 2/math.sqrt(3), 1 - 4/math.sqrt(3)),
            (1 - 2/math.sqrt(3), 1 - 2/math.sqrt(3), 1 + 4/math.sqrt(3))]
for p in page_min:
    ax3.scatter(*p, color=LOW, s=50, depthshade=False)
for p in page_max:
    ax3.scatter(*p, color=GRAD, s=50, depthshade=False)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')
ax3.set_box_aspect((1, 1, 1))
ax3.set_title('The circle on the sphere')

ax = fig.add_subplot(1, 2, 2)
ax.plot(np.degrees(ss[::50]), T_vals[::50], color=FENCE, lw=2.2)
ax.axhline(25, color=LOW, ls='--', lw=1)
ax.axhline(91/3, color=GRAD, ls='--', lw=1)
# Local minima and maxima of the scan (the list wraps around, since the curve is closed).
prev, nxt = np.roll(T_vals, 1), np.roll(T_vals, -1)
for i in np.where((T_vals < prev) & (T_vals <= nxt))[0]:
    ax.plot([np.degrees(ss[i])], [T_vals[i]], 'o', color=LOW, ms=9)
for i in np.where((T_vals > prev) & (T_vals >= nxt))[0]:
    ax.plot([np.degrees(ss[i])], [T_vals[i]], 'o', color=GRAD, ms=9)
ax.set_xlim(0, 360)
ax.set_xticks(range(0, 361, 45))
ax.set_xlabel('s (degrees) around the circle')
ax.set_ylabel('T (°C)')
ax.set_title('T around the circle')
ax.grid(alpha=0.3)
ax.legend(handles=[
    legend_line(CONTOUR, 'grey mesh: the sphere x² + y² + z² = 11'),
    legend_line(FENCE, 'purple: the circle, and T along it'),
    legend_line(LOW, "red: minimum points (dots) and the level T = 25 (dashed)", marker='o'),
    legend_line(GRAD, "blue: maximum points (dots) and the level T = 91/3 (dashed)", marker='o'),
], loc='upper center', bbox_to_anchor=(0.3, -0.16), ncol=2, fontsize=9, frameon=False)
ax3.view_init(elev=28, azim=40)
hud_frame(fig, 'SEC 6 · SPHERE MEETS PLANE', footer='FENCE LINE · T = 20 + 2x + 2y + z² · sphere ∩ plane')
plt.show()

*Figure description.* Left: a grey mesh sphere and the purple circle where it meets the plane. Red dots are the page's two minimum points and blue dots are its two maximum points. All four lie on the circle. Right: the temperature $T$ around the circle, against the angle $s$. The red dashed line is the level $T=25$ and the blue dashed line is the level $T=91/3$. The curve touches the red line twice and the blue line twice.

### 6.1 · Two shadow prices (lecture §3 and §6)

The page reads a multiplier as a sensitivity. If a constraint constant changes by $\Delta c$, the best value changes by about (multiplier)$\,\times\,\Delta c$.
Example 2 has two constants: $11$ for the sphere and $3$ for the plane. So it has two multipliers, $\lambda$ for the sphere and $\mu$ for the plane.

The first cell solves the page's system and prints $\lambda$ and $\mu$ at the four points. The second cell moves one constant at a time by $\Delta=0.01$, scans the new circle, and measures the change of $T_{\min}$ and $T_{\max}$.

In [ ]:
L2, eqs2, sols2, vals2 = lagrange_system(T_expr, [g2 - 11, h2 - 3], [x, y, z], [lam, mu])
print(f"{'point (x, y, z)':<34}{'T':>8}{'lambda':>10}{'mu':>10}")
for s, val in sorted(zip(sols2, vals2), key=lambda p: float(p[1])):
    pt = '(' + ', '.join(fmt(s[v], 4) for v in (x, y, z)) + ')'
    print(f'{pt:<34}{fmt(val, 4):>8}{fmt(s[lam], 4):>10}{fmt(s[mu], 4):>10}')

In [ ]:
delta = 0.01
base_min, base_max = T_vals.min(), T_vals.max()
_, _, T_sphere = circle_scan(c_g=11 + delta)
_, _, T_plane = circle_scan(c_h=3 + delta)

print(f"{'change':<26}{'(new T_min - 25)/delta':>24}{'(new T_max - 91/3)/delta':>26}")
print(f"{'sphere 11 -> 11.01':<26}{(T_sphere.min() - base_min)/delta:>24.4f}{(T_sphere.max() - base_max)/delta:>26.4f}")
print(f"{'plane   3 -> 3.01':<26}{(T_plane.min() - base_min)/delta:>24.4f}{(T_plane.max() - base_max)/delta:>26.4f}")

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🚀 Transfer questions.</strong>

1. Compare the sphere row with the column $\lambda$ of the table. At the two minimum points, $\lambda=0$ (this is Case A on the page). What does $\lambda=0$ predict for $T_{\min}$ when the sphere grows a little? Does the scan agree?
2. Compare the plane row for $T_{\min}$ with $\mu$ at the minimum points. Do they agree?
3. In one sentence: if the plane constant rises from $3$ to $3.01$, by about how much does $T_{\min}$ change, and which multiplier told you so before the scan?

</div>

<details>
<summary>Look closer: the plane row for T_max</summary>

The two maximum points have **different** values of $\mu$, about $-0.87$ and $2.21$. When the plane moves, both points move, and the larger of the two new values becomes the new maximum. So the rate of $T_{\max}$ is close to the larger $\mu$, about $2.21$. Run the cell again with `delta = 0.001` and watch the rate approach it. This only happens because the maximum is attained at two points.

</details>

---
## 7 · CREATE · Mission log

📝 **STAGE 6/7 · CREATE** ▰▰▰▰▰▰▱

Fill in the log with your own readings from Sections 2–6. Decimals with three digits are enough. Exact strings such as `"3/sqrt(2)"` also work.
Then write one sentence in your own words.

In [ ]:
# ✏️ CREATE. Fill in your mission log, then run this cell and the next one.
mission_log = {
    'k_max': None,              # the largest k for which 4xy = k touches the fence (Section 2)
    'vertex': [None, None],     # the contact point [x, y] in the first quadrant (Section 3)
    'lambda_ex1': None,         # the multiplier at that point (Section 4)
    'T_range_ex2': [None, None],  # [T_min, T_max] on the circle of Example 2 (Section 6)
    'one_sentence': '',         # why is the level curve tangent to the fence at the maximum?
}

In [ ]:
# Check the mission log. The card shows only "correct" or "try again", never the expected values.
checks = [
    ('k_max', check_answer(mission_log['k_max'], float(f_star), tol=0.01)),
    ('vertex', check_answer(mission_log['vertex'], [float(x_star), float(y_star)], tol=0.01)),
    ('lambda_ex1', check_answer(mission_log['lambda_ex1'], float(lam_star), tol=0.01)),
    ('T_range_ex2', check_answer(mission_log['T_range_ex2'], [float(min(vals2)), float(max(vals2))], tol=0.01)),
]
rows = [(key, '✅', 'correct') if ok else (key, '🔁', 'try again') for key, ok in checks]
sentence = mission_log['one_sentence'].strip()
rows.append(('one_sentence', '✅', 'logged') if sentence else ('one_sentence', '✏️', 'write one sentence'))

n_ok = sum(ok for _, ok in checks)
mission_log_ok = n_ok == len(checks) and bool(sentence)
summary = f'{n_ok} of {len(checks)} correct.'
summary += ' Sentence logged.' if sentence else ' Sentence not written yet.'
if mission_log_ok:
    summary += ' Mission log complete. Next: the debrief.'
check_card('MISSION LOG · CHECK', rows, summary)
if mission_log_ok and globals().get('quiz_ok', False):
    mission_complete_card()

---
## 8 · DEBRIEF · ⚠️ Common errors: check yourself against these

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#FEF2F2;border-left:6px solid #DC2626;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>⚠️ Common errors.</strong>

- **Forgetting the constraint equation itself.** $\nabla f=\lambda\nabla g$ alone is two equations in three unknowns $(x,y,\lambda)$. You need $g=c$ as the third equation.
- **Stopping at a crossing.** In Section 2, a level curve that cuts the fence at two points is not optimal. A slightly larger $k$ still touches. The optimum is where the curves are tangent.
- **Dividing by a variable without checking if it could be zero.** In Section 5, dividing by $x$ skipped the points $(0,\pm4)$. We had to check them by hand.
- **Stopping once a solution of the system is found.** The system gives candidates. In Section 5 it gave two maxima and two minima. Compare the values of $f$ at all candidates.
- **Treating $\lambda$ as a coordinate to report.** It is not part of the point. It measures the sensitivity of the best value to the constraint constant (Section 6.1).
- **Ignoring the regularity hypothesis.** If $\nabla g=\mathbf 0$ at a point of the constraint curve, the theorem says nothing there. Check that point by hand.

</div>

## 9 · DEBRIEF · Retrieval check: mini-quiz

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

Fill in `student_answers` from memory, then run the cell. Numbers may be decimals (three digits are enough) or exact strings such as `"8*sqrt(2)"`.
The check card shows only **correct** or **try again**. After your second try you may set `reveal = True` to see the key.

In [ ]:
# ✏️ Mini-quiz on Examples 1 and 2 of the Lecture 21 page.
student_answers = {
    'n_equations_ex2': None,                   # number of equations (= unknowns) in the Example 2 system
    'grad_f_at_vertex': None,                  # grad f at the Example 1 maximizer, as [f_x, f_y]
    'lambda_at_T_min_ex2': None,               # the sphere multiplier lambda at the points where T = 25
    'mu_at_T_min_ex2': None,                   # the plane multiplier mu at the points where T = 25
    'tangent_means_parallel_gradients': None,  # True or False: tangent curves at a point <=> parallel gradients there
}
reveal = False   # set to True only after your second try

_key = json.loads(base64.b64decode('eyJuX2VxdWF0aW9uc19leDIiOiA1LCAiZ3JhZF9mX2F0X3ZlcnRleCI6IFsxMS4zMTM3MDg0OTg5ODQ3NjEsIDguNDg1MjgxMzc0MjM4NTddLCAibGFtYmRhX2F0X1RfbWluX2V4MiI6IDAsICJtdV9hdF9UX21pbl9leDIiOiAyLCAidGFuZ2VudF9tZWFuc19wYXJhbGxlbF9ncmFkaWVudHMiOiB0cnVlfQ=='))
if any(v is not None for v in student_answers.values()):
    quiz_attempts = globals().get('quiz_attempts', 0) + 1
else:
    quiz_attempts = globals().get('quiz_attempts', 0)

n_correct = 0
rows = []
for name, target in _key.items():
    ok = check_answer(student_answers[name], target)
    n_correct += ok
    rows.append((name, '✅', 'correct') if ok else (name, '🔁', 'try again'))
quiz_ok = n_correct == len(_key)
check_card('RETRIEVAL · CHECK', rows, f'{n_correct} of {len(_key)} correct · attempt {quiz_attempts}')

if reveal and quiz_attempts < 2:
    print('Try at least twice before you reveal the key.')
elif reveal:
    print('Key:', _key)
elif quiz_ok and globals().get('mission_log_ok', False):
    mission_complete_card()
elif quiz_ok:
    print('All correct. Finish the mission log in Stage 6 to complete the mission.')

## 10 · DEBRIEF · 🤖 AI Learning Companion

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:6px 14px;border-radius:8px;color:#1E293B;">
<strong>🤖 Student-facing AI contract — paste this at the start of every AI session</strong>
<br>1. Ask me what I have already tried before you explain anything.
<br>2. Do not give a complete solution until I have made a prediction or proposed a next step.
<br>3. Point to the first gap in my reasoning, not every later consequence.
<br>4. Make me verify the result by units, sign, substitution, a graph, or a limiting case.
<br>5. End with one transfer question that changes the context.
</div>

<div style="background:#F8FAFC;border-left:6px solid #94A3B8;padding:6px 14px;border-radius:8px;margin-top:6px;color:#1E293B;">
<strong>🔎 Verification rule.</strong> Treat an AI answer as a conjecture. Check it against the definitions, theorems and figures on this page. Bring any disagreement to class. An AI answer never overrides a checked definition or proof. Neither does an illustration.
</div>

**Prompt for this lab: Error Hunter.** Paste your own Lagrange multiplier solution in place of `[paste my work]`.

```
Here is my solution to a Lagrange multiplier problem:
[paste my work]
Find the FIRST invalid step. Do not fix it. Instead:
(1) quote that step, (2) name the underlying misconception in one sentence,
(3) give me a two-line counterexample or sanity check that exposes it,
(4) ask me to repair the step myself.
If my solution is correct, instead give me a harder variant that breaks the
method I used, and ask me why it breaks.
```

## 11 · DEBRIEF · 🔒 Independence check

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:8px 16px;border-radius:8px;color:#1E293B;">
<strong>🔒 Independence rule.</strong> <i>No AI, no hints, new numbers. This shows that you can do it alone.</i>
</div>

Let $f(x,y)=2x+3y$ on the ellipse $x^2+3y^2=7$.

1. Write the Lagrange system: three equations in $(x,y,\lambda)$.
2. Find both solutions of the system.
3. Which one gives the maximum, and which the minimum? What is $\lambda$ at the maximum?

Do this on paper, without AI and without the worked examples. It is not checked here. It is practice for HW8 and Exam 2.

---
## 12 · OPTIONAL · Review examples from the lecture page

**OPTIONAL · outside the seven stages.** Skip this part if you are short of time. The mission ends with the prediction for Lecture 22 below.

### 12.1 · Example 2 · The case split

The page subtracts the first two Lagrange equations and gets $0=2\lambda(y-x)$. So $\lambda=0$ (Case A) or $x=y$ (Case B).
Try both cases on paper first. The cell below solves each case separately with sympy.

In [ ]:
eqs_T = [sp.diff(T_expr, v) - lam*sp.diff(g2, v) - mu*sp.diff(h2, v) for v in (x, y, z)] + [g2 - 11, h2 - 3]
show_text('**Case A** (lambda = 0):')
for s in sp.solve([e.subs(lam, 0) for e in eqs_T], [x, y, z, mu], dict=True):
    show_math(r'(x,y,z)=(' + ',\ '.join(sp.latex(s[v]) for v in (x, y, z)) + r'),\quad \mu=' + sp.latex(s[mu])
              + r',\quad T=' + sp.latex(sp.simplify(T_expr.subs(s))))
show_text('**Case B** (x = y, lambda not 0):')
for s in sp.solve([e.subs(y, x) for e in eqs_T], [x, z, lam, mu], dict=True):
    if s[lam] != 0:
        show_math(r'(x,y,z)=(' + ',\ '.join(sp.latex(sp.radsimp(s[v])) for v in (x, x, z)) + r'),\quad \lambda='
                  + sp.latex(s[lam]) + r',\quad T=' + sp.latex(sp.nsimplify(sp.simplify(T_expr.subs({x: s[x], y: s[x], z: s[z]})))))

### 12.2 · The Lagrangian form (lecture §7)

The page writes $\mathcal L(x,y,\lambda)=f(x,y)-\lambda\,(g(x,y)-c)$. Setting $\nabla_{x,y}\mathcal L=\mathbf 0$ gives $\nabla f=\lambda\nabla g$. Setting $\partial\mathcal L/\partial\lambda=0$ gives back the constraint $g=c$.
The cell checks the second fact for Example 1.

In [ ]:
show_math(r'\frac{\partial \mathcal L}{\partial \lambda} = ' + sp.latex(sp.diff(L1, lam)))
print('dL/dlambda = 0 is the fence equation g = 1:', sp.simplify(sp.diff(L1, lam) + (g_expr - c_fence)) == 0)

---
## 13 · DEBRIEF · Prediction for Lecture 22

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

The mission is complete when the Stage 6 log and the retrieval check both show ✅ for every item.
Your last task is one prediction.

<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:10px;padding:10px 16px;color:#E2E8F0;">
<span style="color:#22D3EE;font-family:Menlo,Consolas,monospace;letter-spacing:1px;font-size:12px;">MISSION 21 · FINAL PROMPT · PREDICTION FOR LECTURE 22</span><br>
Chapter 14 asks how to add up $f$ over a region instead of differentiating it.
How would you estimate the area of the ellipse from lecture §4 by slicing it into thin strips?
<br>Write your idea in one sentence. Lecture 22 will test it.
</div>

---
<div style="background:#0F1A2B;border-top:2px solid #22D3EE;color:#94A3B8;padding:8px 16px;font-family:Menlo,Consolas,monospace;font-size:12px;letter-spacing:1px;">
END OF LAB 21 · FENCE LINE · © Tan H. Cao, for AMS 261 students
</div>